# Quickstart: alanine dipeptide, from trajectories to macrostates

This notebook builds a complete Markov state model (MSM) with MSMBuilder Neo:

1. load trajectories and compute features,
2. cluster the frames into microstates,
3. choose a lag time from the implied timescales and fit the MSM,
4. validate it (cross-validated VAMP score, Chapman-Kolmogorov test),
5. group the microstates into PCCA+ macrostates,
6. compute macrostate populations, mean first passage times and fluxes,
7. write representative structures of each macrostate as PDB files.

It uses the alanine dipeptide example dataset (ten 10 ns trajectories, downloaded once).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

from msmbuilder.example_datasets import AlanineDipeptide

data = AlanineDipeptide().get()
trajectories = data.trajectories
print(len(trajectories), 'trajectories of', trajectories[0].n_frames, 'frames,',
      trajectories[0].n_atoms, 'atoms; time step', trajectories[0].timestep, 'ps')

## Features

The backbone dihedrals $\phi$ and $\psi$ describe the conformations of alanine dipeptide.
Every MSMBuilder estimator takes a *list* of trajectories and returns a list.

In [ ]:
from msmbuilder.featurizer import DihedralFeaturizer

featurizer = DihedralFeaturizer(types=['phi', 'psi'], sincos=False)
dihedrals = featurizer.fit_transform(trajectories)
allx = np.degrees(np.concatenate(dihedrals))

plt.hist2d(allx[:, 0], allx[:, 1], bins=90, cmap='viridis', norm='log')
plt.xlabel(r'$\phi$ (degrees)')
plt.ylabel(r'$\psi$ (degrees)')
plt.title('Ramachandran plot')
plt.colorbar(label='Frames');

## Microstates

Cluster the frames in the sine/cosine representation of the dihedrals, so that
$-180^\circ$ and $180^\circ$ are close.

In [ ]:
from msmbuilder.cluster import KCenters

sincos = DihedralFeaturizer(types=['phi', 'psi'], sincos=True).fit_transform(trajectories)
clusterer = KCenters(n_clusters=100, random_state=0)
labels = clusterer.fit_transform(sincos)
print('frame labels of the first trajectory:', labels[0][:10])

## Lag time and Markov state model

The implied timescales of MSMs estimated at increasing lag times level off once the
model is Markovian; pick a lag time in that range.

In [ ]:
from msmbuilder.msm import MarkovStateModel, implied_timescales

dt_ps = trajectories[0].timestep
lag_times = [1, 2, 5, 10, 20, 30]
its = implied_timescales(labels, lag_times, n_timescales=3, msm=MarkovStateModel(verbose=False))

plt.semilogy(np.array(lag_times) * dt_ps, its * dt_ps, 'o-')
plt.xlabel('Lag time (ps)')
plt.ylabel('Implied timescale (ps)')
plt.title('Implied timescales');

In [ ]:
lag = 10
msm = MarkovStateModel(lag_time=lag, verbose=False).fit(labels)
print(f'{msm.n_states_} states; slowest timescales: {msm.timescales_[:3] * dt_ps} ps')

## Validation

**Cross-validated VAMP-2 score.** Fit on some trajectories, score on the others: a better
model of the slow dynamics scores higher (`score_method='gmrq'`, the default, is also available).

In [ ]:
from sklearn.model_selection import KFold

scores = []
for train, test in KFold(n_splits=5, shuffle=True, random_state=0).split(labels):
    model = MarkovStateModel(lag_time=lag, n_timescales=2, score_method='vamp2', verbose=False)
    model.fit([labels[i] for i in train])
    scores.append(model.score([labels[i] for i in test]))
print('held-out VAMP-2 score: %.3f +/- %.3f' % (np.mean(scores), np.std(scores)))

**Chapman-Kolmogorov test** (needs `deeptime`): the model at lag $\tau$, propagated
$k$ times, should predict the models estimated at lag $k\tau$.

In [ ]:
try:
    from msmbuilder.msm import ck_test
    test = ck_test(MarkovStateModel(verbose=False), labels, [lag, 2 * lag, 4 * lag, 8 * lag],
                   n_metastable_sets=2)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    for k, ax in enumerate(axes):
        ax.plot(test.lagtimes * dt_ps, test.predictions[:, k, k], 'k--', label='predicted')
        ax.plot(test.lagtimes * dt_ps, test.estimates[:, k, k], 'o', label='estimated')
        ax.set_title(f'set {k} -> set {k}')
        ax.set_xlabel('Lag time (ps)')
    axes[0].legend()
    plt.tight_layout()
except ImportError:
    print('Install deeptime for the Chapman-Kolmogorov test: pip install deeptime')

## Macrostates with PCCA+

PCCA+ groups the microstates into metastable macrostates, with fuzzy memberships.
`method='deeptime'` uses PyEMMA's algorithm (and needs deeptime); the default method needs
nothing extra.

In [ ]:
from msmbuilder.lumping import PCCAPlus

try:
    import deeptime  # noqa: F401
    pcca = PCCAPlus.from_msm(msm, n_macrostates=2, method='deeptime')
except ImportError:
    pcca = PCCAPlus.from_msm(msm, n_macrostates=2, random_state=0)

print('macrostate populations (memberships):', pcca.coarse_grained_populations_.round(3))
print('macrostate transition matrix at lag', lag * dt_ps, 'ps:')
print(pcca.coarse_grained_transmat_.round(4))

centers = np.degrees(np.arctan2(clusterer.cluster_centers_[:, :2], clusterer.cluster_centers_[:, 2:]))
macro_of_label = {label: pcca.microstate_mapping_[i] for label, i in msm.mapping_.items()}
plt.hist2d(allx[:, 0], allx[:, 1], bins=90, cmap='Greys', norm='log')
for label, (phi, psi) in enumerate(centers):
    if label in macro_of_label:
        plt.scatter(phi, psi, c=f'C{macro_of_label[label]}', s=40, edgecolors='k')
plt.xlabel(r'$\phi$ (degrees)')
plt.ylabel(r'$\psi$ (degrees)')
plt.title('Microstates colored by macrostate');

## Populations, first passage times and fluxes between macrostates

The transition path theory functions in `msmbuilder.tpt` work on sets of microstates, so
they take the PCCA+ macrostates directly.

In [ ]:
from msmbuilder import tpt

pi = msm.populations_
A, B = pcca.metastable_sets_

# mean first passage time A -> B: average over A, weighted by the stationary distribution
mfpt_to_B = tpt.mfpts(msm, sinks=list(B), lag_time=lag * dt_ps)
mfpt_to_A = tpt.mfpts(msm, sinks=list(A), lag_time=lag * dt_ps)
print('MFPT A -> B: %.0f ps' % ((pi[A] * mfpt_to_B[A]).sum() / pi[A].sum()))
print('MFPT B -> A: %.0f ps' % ((pi[B] * mfpt_to_A[B]).sum() / pi[B].sum()))

# reactive flux A -> B and the transition rate (per lag time)
net_flux = tpt.net_fluxes(list(A), list(B), msm)
committor = tpt.committors(list(A), list(B), msm)
total_flux = net_flux[A].sum()
rate = total_flux / np.dot(pi, 1 - committor)
print('total reactive flux A -> B: %.4g per lag time; rate %.3g per ps' % (total_flux, rate / (lag * dt_ps)))

## Representative structures of each macrostate

`save_macrostate_samples` draws frames from each macrostate (like PyEMMA's
`sample_by_distributions`) and writes them as one multi-model PDB file per macrostate,
superposed on the backbone, with a CSV listing where every structure came from.

In [ ]:
import os, tempfile
from msmbuilder.lumping import save_macrostate_samples

out_dir = tempfile.mkdtemp()
result = save_macrostate_samples(pcca, labels, trajectories, n_samples=20,
                                 superpose='backbone', out_dir=out_dir, random_state=0)
print([os.path.basename(f) for f in result['files']])
result['index'].head()

In [ ]:
for k, path in enumerate(result['files']):
    sample = md.load(path)
    phi = np.degrees(md.compute_phi(sample)[1][:, 0])
    psi = np.degrees(md.compute_psi(sample)[1][:, 0])
    plt.scatter(phi, psi, label=f'macrostate {k}', edgecolors='k')
plt.xlim(-180, 180)
plt.ylim(-180, 180)
plt.xlabel(r'$\phi$ (degrees)')
plt.ylabel(r'$\psi$ (degrees)')
plt.legend()
plt.title('Dihedrals of the saved structures');

## Next steps

* The same analysis can be run from one YAML file with `msmb Pipeline`, which also records
  provenance: see {ref}`pipeline`.
* Nonlinear slow coordinates (VAMPnets, SRVs, DeepTICA) are described under
  {ref}`decomposition`.
* For biased simulations, see TRAM under {ref}`msm`.